In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [2]:
X_train_scaled = pd.read_csv("../data/processed/X_train_scaled.csv")
X_test_scaled = pd.read_csv("../data/processed/X_test_scaled.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [3]:
X_train = pd.read_csv("../data/processed/X_train_final.csv")
X_test = pd.read_csv("../data/processed/X_test_final.csv")

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix

solver_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky']

best_solver = None
best_score = 0

print("Comparaison des solveurs :")

for s in solver_list:
    
    lr = LogisticRegression(
        solver=s,
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    )
    
    scores = cross_val_score(
        lr,
        X_train_scaled,
        y_train,
        cv=5,
        scoring='recall'
    )
    
    mean_score = scores.mean()
    
    print(
        f"Solver {s:18s} → "
        f"Recall moyen : {mean_score:.4f}"
    )
    
    if mean_score > best_score:
        best_score = mean_score
        best_solver = s

print(f"\nMeilleur solveur : {best_solver}")
print(f"Meilleur Recall CV : {best_score:.4f}")

lr_final = LogisticRegression(
    solver=best_solver,
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

lr_final.fit(X_train_scaled, y_train)

y_pred = lr_final.predict(X_test_scaled)

print("\n ÉVALUATION SUR LE TEST : ")

print(   f"Recall    : {recall_score(y_test, y_pred):.4f}")

print( f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")

print(
    "Matrice de confusion :\n",
    confusion_matrix(y_test, y_pred)
)

Comparaison des solveurs :
Solver lbfgs              → Recall moyen : 0.8521
Solver liblinear          → Recall moyen : 0.8496
Solver newton-cg          → Recall moyen : 0.8521
Solver newton-cholesky    → Recall moyen : 0.8521

Meilleur solveur : lbfgs
Meilleur Recall CV : 0.8521

 ÉVALUATION SUR LE TEST : 
Recall    : 0.8725
Accuracy  : 0.8750
Matrice de confusion :
 [[72 10]
 [13 89]]


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import recall_score, confusion_matrix, accuracy_score

rfc = RandomForestClassifier(
    class_weight='balanced',
    random_state=42
)

param_grid = {
    'n_estimators': [100, 200],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [4, 6, 8, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    rfc,
    param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Meilleurs paramètres :", grid_search.best_params_)

best_rf = RandomForestClassifier(
    **grid_search.best_params_,
    class_weight='balanced',
    random_state=42
)

best_rf.fit(X_train, y_train)

y_pred = best_rf.predict(X_test)

print("\n ÉVALUATION SUR LE TEST : ")

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print(
    "Matrice de confusion :\n",
    confusion_matrix(y_test, y_pred)
)

Meilleurs paramètres : {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}

 ÉVALUATION SUR LE TEST : 
Accuracy : 0.8152173913043478
Recall   : 0.8235294117647058
Matrice de confusion :
 [[66 16]
 [18 84]]


In [6]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix

xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

print("Meilleurs paramètres XGBoost :")
print(grid_xgb.best_params_)

print(f"\nMeilleur Recall CV : {grid_xgb.best_score_:.4f}")

best_xgb = grid_xgb.best_estimator_

y_pred_xgb = best_xgb.predict(X_test)

print("\n ÉVALUATION SUR LE TEST : ")

print(  f"Recall   : {recall_score(y_test, y_pred_xgb):.4f}")

print(f"Accuracy : {accuracy_score(y_test, y_pred_xgb):.4f}")

print(
    "Matrice de confusion :\n",
    confusion_matrix(y_test, y_pred_xgb)
)

Meilleurs paramètres XGBoost :
{'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 100, 'subsample': 0.8}

Meilleur Recall CV : 0.9137

 ÉVALUATION SUR LE TEST : 
Recall   : 0.8627
Accuracy : 0.8478
Matrice de confusion :
 [[68 14]
 [14 88]]


In [7]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    recall_score,
    accuracy_score,
    confusion_matrix,
)

svm = SVC(
    probability=True,
    random_state=42
)

param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

grid_svm.fit(X_train_scaled, y_train)

print("Meilleurs paramètres SVM :")
print(grid_svm.best_params_)

print(f"\nMeilleur Recall moyen CV : {grid_svm.best_score_:.4f}")


best_svm = grid_svm.best_estimator_

best_svm.fit(X_train_scaled, y_train)

y_pred_svm = best_svm.predict(X_test_scaled)

y_prob_svm = best_svm.predict_proba(X_test_scaled)[:, 1]

print("\n ÉVALUATION SUR LE TEST : ")

print(f"Accuracy  : {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_svm):.4f}")

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_svm))

Meilleurs paramètres SVM :
{'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}

Meilleur Recall moyen CV : 0.9014

 ÉVALUATION SUR LE TEST : 
Accuracy  : 0.8587
Recall    : 0.9216

Matrice de confusion :
[[64 18]
 [ 8 94]]


In [8]:
import joblib

joblib.dump(lr_final, '../models/lr_final.pkl')     
joblib.dump(best_svm, '../models/best_svm.pkl')  
joblib.dump(best_rf, '../models/best_rf.pkl')
joblib.dump(best_xgb, '../models/best_xgb.pkl')

['../models/best_xgb.pkl']